In [1]:
pip install nltk torch

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
from nltk.util import ngrams
from collections import defaultdict, Counter
import torch
import torch.nn as nn

In [3]:
# --------------------------------------------------
# Step 1: Corpus
# --------------------------------------------------

corpus = (
    "machine learning enables computers to learn from data "
    "deep learning is a subset of machine learning "
    "large language models use tokenization techniques "
    "artificial intelligence is transforming industries "
    "transformers are widely used in natural language processing "
    "machine learning models require large amounts of data "
    "deep learning models are used in natural language processing"
)

words = corpus.split()
print("Total words in corpus:", len(words))

Total words in corpus: 52


In [4]:
# Step 2: Build a Trigram N-gram Model
# --------------------------------------------------

trigrams = list(ngrams(words, 3))
trigram_counts = defaultdict(Counter)

for w1, w2, w3 in trigrams:
    trigram_counts[(w1, w2)][w3] += 1


def ngram_predict(w1, w2, top_k=3):
    candidates = trigram_counts.get((w1, w2))
    if not candidates:
        return []
    return candidates.most_common(top_k)


seed = ("machine", "learning")
print("\n===== N-gram (Trigram) Prediction =====")
print(f"Given: '{seed[0]} {seed[1]}' ->")
print(ngram_predict(*seed))


===== N-gram (Trigram) Prediction =====
Given: 'machine learning' ->
[('enables', 1), ('large', 1), ('models', 1)]


In [5]:
# --------------------------------------------------
# Step 3: Build a Tiny LSTM Language Model
# --------------------------------------------------

vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)

data = [(w1, w2, w3) for w1, w2, w3 in trigrams]

In [6]:
class TinyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)


In [7]:
model = TinyLSTM(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

X = torch.tensor([[word_to_idx[w1], word_to_idx[w2]] for w1, w2, w3 in data])
y = torch.tensor([word_to_idx[w3] for w1, w2, w3 in data])

print("\n===== Training Tiny LSTM =====")
for epoch in range(200):
    optimizer.zero_grad()
    output = model(X)
    loss = loss_fn(output, y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")



===== Training Tiny LSTM =====
Epoch 50, Loss: 0.1420
Epoch 100, Loss: 0.1263
Epoch 150, Loss: 0.1244
Epoch 200, Loss: 0.1234


In [8]:
def lstm_predict(w1, w2, top_k=3):
    x = torch.tensor([[word_to_idx[w1], word_to_idx[w2]]])
    with torch.no_grad():
        logits = model(x)[0]
    top = torch.topk(logits, top_k)
    return [(idx_to_word[idx.item()], round(score.item(), 2))
            for score, idx in zip(top.values, top.indices)]


print("\n===== LSTM Prediction =====")
print(f"Given: '{seed[0]} {seed[1]}' ->")
print(lstm_predict(*seed))


===== LSTM Prediction =====
Given: 'machine learning' ->
[('models', 9.79), ('large', 9.79), ('enables', 9.79)]


In [9]:
# --------------------------------------------------
# Step 4: Compare Both Models
# --------------------------------------------------

print("\n===== Comparison =====")
print("N-gram top prediction:", ngram_predict(*seed)[:1])
print("LSTM top prediction  :", lstm_predict(*seed)[:1])



===== Comparison =====
N-gram top prediction: [('enables', 1)]
LSTM top prediction  : [('models', 9.79)]
